# General Instructions:
You will utilize the "Medical Cost Personal Datasets". Ensure that you have downloaded this dataset.

Employ the Python programming language and appropriate libraries such as NumPy, Pandas, Matplotlib, and Scikit-Learn to perform regression analysis and SVR.

# Tasks
Identify the variables to be used as independent variables (features) and the target variable (personal medical cost).

Partition the dataset into training and testing subsets with an appropriate proportion.

Perform feature scaling if required.

Construct a multiple linear regression model using Scikit-Learn.

Train the model on the training data and perform predictions on the testing data.

Evaluate the model by computing metrics such as R-squared, MSE, and MAE. Present the evaluation results.

Repeat step 4 using an SVR model. You may experiment by performing hyperparameter tuning.

# Additional Notes
Ensure to provide a concise description of the dataset used, including descriptions of relevant variables.

Present relevant data visualizations such as scatter plots or other graphs if necessary for improved understanding.

Analyze the results obtained in your practical assignment.

Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

df = pd.read_csv('insurance.csv')

print("Dataset Overview:")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())

Dataset Overview:
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 94.5 KB
None

First 5 rows:
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


# Step 1: Identify Independent Variables (Features) and Target Variable

In [2]:
# 1. Load dataset
df = pd.read_csv('insurance.csv')

# 2. Define features (X) and target variable (y)
X = df[['age', 'sex', 'bmi', 'children', 'smoker', 'region']]
y = df['charges']

print('Features (X):', list(X.columns))
print('Target (y):', y.name)

Features (X): ['age', 'sex', 'bmi', 'children', 'smoker', 'region']
Target (y): charges


# Step 2: Partition Dataset into Training and Testing Subsets
The dataset is partitioned into an 80% training set (1,070 samples) and a 20% testing set (268 samples) with random_state = 42 to ensure reproducible results.

In [3]:
# Split dataset into training and testing sets (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set size: {X_train.shape[0]} samples')
print(f'Testing set size : {X_test.shape[0]} samples')

Training set size: 1070 samples
Testing set size : 268 samples


# Step 3: Perform Feature Scaling and Encoding
Categorical Features (sex, smoker, region): Converted into numerical representation using One-Hot Encoding (drop='first') to avoid multicollinearity. Numerical Features (age, bmi, children): Scaled using StandardScaler, ensuring features are on a uniform scale for distance-sensitive algorithms like SVR.   

In [5]:
numerical_cols = ['age', 'bmi', 'children']
categorical_cols = ['sex', 'smoker', 'region']

# Construct preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first'), categorical_cols),
    ]
)

# Fit on training data and transform both train and test sets
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

# Step 4: Construct Multiple Linear Regression Model
A Multiple Linear Regression model is instantiated using Scikit-Learn's LinearRegression module.

In [6]:
# Instantiate Multiple Linear Regression model
lr_model = LinearRegression()

# Step 5: Train Model and Perform Predictions
The model is trained on the preprocessed training set (X_train_scaled, y_train) and evaluated by predicting medical charges on the unseen testing set (X_test_scaled).

In [7]:
# Train model on training data
lr_model.fit(X_train_scaled, y_train)

# Predict charges on testing data
y_pred_lr = lr_model.predict(X_test_scaled)

# Step 6: Evaluate Multiple Linear Regression Model
The performance of the Multiple Linear Regression model is evaluated using R-squared, Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and Mean Absolute Error (MAE).

In [8]:
r2_lr = r2_score(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print('--- Multiple Linear Regression Evaluation ---')
print(f'R-squared (R2) : {r2_lr:.4f}')
print(f'MSE            : {mse_lr:.2f}')
print(f'RMSE           : {rmse_lr:.2f}')
print(f'MAE            : {mae_lr:.2f}')

--- Multiple Linear Regression Evaluation ---
R-squared (R2) : 0.7836
MSE            : 33596915.85
RMSE           : 5796.28
MAE            : 4181.19


# Step 7: Support Vector Regression (SVR) Model & Hyperparameter Tuning
Steps 4–6 are repeated using a Support Vector Regression (SVR) model combined with 5-fold Cross-Validation Grid Search (GridSearchCV) to optimize hyperparameters.

In [9]:
# Define hyperparameter grid search space
param_grid = {
    'C': [1000, 5000, 10000, 20000, 50000],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf'],
}

# Perform grid search cross-validation
grid_svr = GridSearchCV(SVR(), param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_svr.fit(X_train_scaled, y_train)

# Select best model and predict
best_svr = grid_svr.best_estimator_
y_pred_svr = best_svr.predict(X_test_scaled)

# Evaluate SVR model
r2_svr = r2_score(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(mse_svr)
mae_svr = mean_absolute_error(y_test, y_pred_svr)

print('--- Support Vector Regression (SVR) Evaluation ---')
print(f'Best Parameters : {grid_svr.best_params_}')
print(f'R-squared (R2)  : {r2_svr:.4f}')
print(f'MSE             : {mse_svr:.2f}')
print(f'RMSE            : {rmse_svr:.2f}')
print(f'MAE             : {mae_svr:.2f}')

--- Support Vector Regression (SVR) Evaluation ---
Best Parameters : {'C': 50000, 'gamma': 'auto', 'kernel': 'rbf'}
R-squared (R2)  : 0.8576
MSE             : 22101172.70
RMSE            : 4701.19
MAE             : 1708.10
